# LeetCode 3014 — Minimum Number of Pushes to Type Word I bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-pushes-to-type-word-i/ · Độ khó: **Easy** · Chủ đề: Math, String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minimumPushes(self, word: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với quy hoạch động vét cạn), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bàn phím điện thoại cũ có **8 phím** (`2` đến `9`); ta được **gán lại** các chữ cái lên các phím tùy ý; **chữ thứ `k` trên một phím cần `k` lần bấm**; tìm cách gán để **tổng số lần bấm** gõ hết từ là nhỏ nhất.
2. Nhận ra đây là bài **tham lam (greedy)**: đặt các chữ vào những vị trí **rẻ nhất** trước, tức **chia đều** các chữ cho 8 phím.
3. Biết dùng **phép chia nguyên `//`** và **chia dư `%`** (và `divmod`) để suy ra "chữ thứ `i` nằm ở lớp nào, tốn bao nhiêu lần bấm": `i // 8 + 1`.
4. Đi từ cách **mô phỏng chia chữ cho từng phím** → **cộng theo từng lớp 8 chữ** → **công thức đóng** → **gói gọn thành một biểu thức `sum(i // 8 + 1 ...)`**.
5. Tránh các lỗi kinh điển: dùng **9 hoặc 10 phím** thay vì 8, quên `+ 1` (chữ đầu tiên cần `1` lần bấm, không phải `0`), chỉ xét trường hợp `n <= 8`, và nhầm công thức `ceil`/`floor` làm lệch một đơn vị.

## 2. Đề bài & ví dụ

Bạn được cho một chuỗi `word` gồm các **chữ cái tiếng Anh thường, đôi một khác nhau** (không có chữ nào lặp lại).

Trên bàn phím điện thoại cũ, các phím được đánh số từ `2` đến `9` (tức **8 phím** để gán chữ). Bạn được **gán lại** các chữ cái cho các phím: **mỗi chữ cái được gán đúng cho một phím**, và mỗi phím có thể chứa **bất kỳ số lượng chữ cái nào** (kể cả không có chữ nào). Để gõ một chữ cái, bạn bấm phím chứa nó: **chữ cái đứng thứ `k` trên phím** cần bấm phím đó **`k` lần**.

Hãy trả về **số lần bấm phím tối thiểu** cần để gõ hết chuỗi `word` (nếu ta chọn cách gán chữ cái tốt nhất).

**Ví dụ 1**

- Input: `word = "abcde"`
- Output: `5`
- Giải thích: `5` chữ cái đặt lên `5` phím khác nhau, mỗi chữ là chữ đầu của phím nên cần `1` lần bấm. Tổng `5`.

**Ví dụ 2**

- Input: `word = "xycdefghij"`
- Output: `12`
- Giải thích: `10` chữ cái, `8` phím. Cách tốt nhất: đặt `8` chữ cái lên `8` phím (mỗi chữ `1` lần bấm, tổng `8`), `2` chữ còn lại đặt làm chữ thứ hai trên hai phím (mỗi chữ `2` lần bấm, tổng `4`). Tổng `8 + 4 = 12`.

**Ràng buộc:** `1 <= word.length <= 26`; `word` chỉ gồm các chữ cái tiếng Anh thường; **các chữ cái trong `word` đôi một khác nhau**.

In [1]:
import random
from itertools import accumulate

examples = [
    ("abcde", 5),
    ("xycdefghij", 12),
    ("a", 1),
    ("abcdefgh", 8),
    ("abcdefghi", 10),
]
for word, expected in examples:
    print("word =", repr(word), "độ dài", len(word), "-> kỳ vọng", expected)

word = 'abcde' độ dài 5 -> kỳ vọng 5
word = 'xycdefghij' độ dài 10 -> kỳ vọng 12
word = 'a' độ dài 1 -> kỳ vọng 1
word = 'abcdefgh' độ dài 8 -> kỳ vọng 8
word = 'abcdefghi' độ dài 9 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Mô hình: 8 phím, chữ thứ `k` tốn `k` lần bấm

Hình dung mỗi phím là một **chồng chữ**: chữ ở vị trí `1` (trên cùng) tốn `1` lần bấm, vị trí `2` tốn `2` lần, vị trí `3` tốn `3` lần, ... Một phím chứa `m` chữ tốn tổng `1 + 2 + ... + m = m(m + 1) / 2` lần bấm (nếu mỗi chữ được gõ một lần). **Thứ tự** các chữ trong một phím không ảnh hưởng tổng này, vì mỗi vị trí `k` đều tốn `k` lần.

In [2]:
def key_cost(m):
    return m * (m + 1) // 2                 # 1 + 2 + ... + m

for m in range(0, 6):
    print(m, "chữ trên một phím tốn", key_cost(m), "lần bấm =", sum(range(1, m + 1)))

0 chữ trên một phím tốn 0 lần bấm = 0
1 chữ trên một phím tốn 1 lần bấm = 1
2 chữ trên một phím tốn 3 lần bấm = 3
3 chữ trên một phím tốn 6 lần bấm = 6
4 chữ trên một phím tốn 10 lần bấm = 10
5 chữ trên một phím tốn 15 lần bấm = 15


### 3.2. Phép chia nguyên `//` và chia dư `%`

`a // b` là thương nguyên, `a % b` là phần dư; `divmod(a, b)` trả cả hai. Với bài này: chữ thứ `i` (đếm từ `0`) nằm ở **lớp** `i // 8` (lớp `0` là vị trí đầu của 8 phím, lớp `1` là vị trí thứ hai, ...), nên tốn **`i // 8 + 1`** lần bấm.

In [3]:
print(17 // 8, 17 % 8, divmod(17, 8))       # 2 1 (2, 1)
for i in [0, 7, 8, 15, 16, 25]:
    print(f"chữ thứ {i}: lớp {i // 8} -> tốn {i // 8 + 1} lần bấm")

2 1 (2, 1)
chữ thứ 0: lớp 0 -> tốn 1 lần bấm
chữ thứ 7: lớp 0 -> tốn 1 lần bấm
chữ thứ 8: lớp 1 -> tốn 2 lần bấm
chữ thứ 15: lớp 1 -> tốn 2 lần bấm
chữ thứ 16: lớp 2 -> tốn 3 lần bấm
chữ thứ 25: lớp 3 -> tốn 4 lần bấm


### 3.3. Tại sao "chia đều" là tối ưu? (trực giác tham lam)

Vì chữ ở vị trí `k` tốn `k` lần bấm, ta muốn **dùng nhiều vị trí rẻ** (vị trí `1`) trước. Có `8` phím nên có `8` vị trí rẻ nhất (cost `1`), tiếp theo `8` vị trí cost `2`, v.v. Với `n` chữ, ta **lấp đầy các vị trí rẻ nhất trước**: `8` chữ đầu cost `1`, `8` chữ tiếp theo cost `2`, ... Đây cũng chính là **chia đều** số chữ cho `8` phím (chênh nhau tối đa `1`).

In [4]:
n = 26
costs = [i // 8 + 1 for i in range(n)]
print(costs)
print("tổng:", sum(costs))

[1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4]
tổng: 56


### 3.4. Tổng một dãy bằng `sum` và biểu thức sinh (generator)

`sum(i // 8 + 1 for i in range(n))` cộng các chi phí của `n` chữ mà không tạo danh sách trung gian. Có thể kiểm tra từng bước bằng cách in danh sách chi phí.

In [5]:
n = 10
print(sum(i // 8 + 1 for i in range(n)))        # 12
print(list(accumulate(i // 8 + 1 for i in range(n))))      # tổng cộng dồn sau mỗi chữ

12
[1, 2, 3, 4, 5, 6, 7, 8, 10, 12]


### 3.5. Số phần tử đầy đủ của từng lớp: `n // 8` và `n % 8`

Với `n` chữ: có `full = n // 8` lớp **đầy đủ** `8` chữ (chi phí lần lượt `1, 2, ..., full`), và `rem = n % 8` chữ **dư** ở lớp kế tiếp (chi phí `full + 1`). Tổng chi phí các lớp đầy đủ: `8 · (1 + 2 + ... + full) = 8 · full(full + 1) / 2 = 4 · full(full + 1)`.

In [6]:
for n in [5, 8, 10, 16, 20, 26]:
    full, rem = divmod(n, 8)
    total = 4 * full * (full + 1) + rem * (full + 1)
    print(f"n={n:>2}: full={full}, rem={rem} -> {4 * full * (full + 1)} + {rem * (full + 1)} = {total}")

n= 5: full=0, rem=5 -> 0 + 5 = 5
n= 8: full=1, rem=0 -> 8 + 0 = 8
n=10: full=1, rem=2 -> 8 + 4 = 12
n=16: full=2, rem=0 -> 24 + 0 = 24
n=20: full=2, rem=4 -> 24 + 12 = 36
n=26: full=3, rem=2 -> 48 + 8 = 56


### 3.6. Chuỗi: `len(word)` là đủ

Vì các chữ cái của `word` **đôi một khác nhau**, mỗi chữ chỉ xuất hiện **một lần** trong lúc gõ; đáp án **chỉ phụ thuộc `len(word)`**, không phụ thuộc chữ cái cụ thể nào. Nếu chữ cái có thể lặp (bản II), ta mới cần xét tần suất.

In [7]:
print(len("abcde"), len("xycdefghij"))        # 5 10
print(len(set("abcde")) == len("abcde"))      # True: các chữ cái đôi một khác nhau

5 10
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `word = "xycdefghij"` (`n = 10`), ta xếp `10` chữ lên `8` phím **lần lượt từng chữ**, đặt chữ vào phím đang có **ít chữ nhất**:

| chữ thứ | phím nhận | vị trí trên phím | lần bấm |
|---|---|---|---|
| 1..8 | phím 1..8 | 1 | 1 mỗi chữ (tổng 8) |
| 9 | phím 1 | 2 | 2 |
| 10 | phím 2 | 2 | 2 |

Tổng `8 · 1 + 2 · 2 = 12`. Nếu thay vì thế dồn nhiều chữ vào một phím, ví dụ đặt `10` chữ trên chỉ `1` phím, ta sẽ tốn `1 + 2 + ... + 10 = 55`, tệ hơn nhiều. **Chia đều là tốt nhất.** Công thức theo từng chữ: chữ thứ `i` (đếm từ `0`) tốn `i // 8 + 1`.

In [8]:
n = 10
keys = [[] for _ in range(8)]
total = 0
for t in range(n):
    k = min(range(8), key=lambda x: len(keys[x]))    # phím đang ít chữ nhất
    keys[k].append(t)
    cost = len(keys[k])                              # vị trí trên phím = số chữ hiện có
    total += cost
    print(f"chữ {t + 1:>2} -> phím {k + 1} (vị trí {cost}, tốn {cost}) | tổng={total}")
print("các phím:", keys)

chữ  1 -> phím 1 (vị trí 1, tốn 1) | tổng=1
chữ  2 -> phím 2 (vị trí 1, tốn 1) | tổng=2
chữ  3 -> phím 3 (vị trí 1, tốn 1) | tổng=3
chữ  4 -> phím 4 (vị trí 1, tốn 1) | tổng=4
chữ  5 -> phím 5 (vị trí 1, tốn 1) | tổng=5
chữ  6 -> phím 6 (vị trí 1, tốn 1) | tổng=6
chữ  7 -> phím 7 (vị trí 1, tốn 1) | tổng=7
chữ  8 -> phím 8 (vị trí 1, tốn 1) | tổng=8
chữ  9 -> phím 1 (vị trí 2, tốn 2) | tổng=10
chữ 10 -> phím 2 (vị trí 2, tốn 2) | tổng=12
các phím: [[0, 8], [1, 9], [2], [3], [4], [5], [6], [7]]


## 5. Cách 1 — Brute Force / baseline: mô phỏng gán từng chữ vào phím ít chữ nhất

**Ý tưởng.** Dùng mảng `size` gồm `8` phần tử (số chữ trên mỗi phím). Với mỗi chữ trong `word`: chọn phím có `size` nhỏ nhất, tăng `size` của nó lên `1` (chữ vừa đặt ở vị trí `size`), cộng `size` vào kết quả.

In [9]:
def minimum_pushes_simulate(word: str) -> int:
    size = [0] * 8                                     # size[k] = số chữ đang có trên phím k
    total = 0
    for _ in word:                                     # mỗi chữ cái (đôi một khác nhau)
        k = size.index(min(size))                      # phím đang ít chữ nhất
        size[k] += 1
        total += size[k]                               # vị trí trên phím = số lần bấm
    return total


for word, expected in examples:
    print(repr(word), "->", minimum_pushes_simulate(word), "(kỳ vọng", expected, ")")

'abcde' -> 5 (kỳ vọng 5 )
'xycdefghij' -> 12 (kỳ vọng 12 )
'a' -> 1 (kỳ vọng 1 )
'abcdefgh' -> 8 (kỳ vọng 8 )
'abcdefghi' -> 10 (kỳ vọng 10 )


**Độ phức tạp (Cách 1).** Mỗi chữ tìm `min` và `index` trên mảng cố định `8` phần tử (`O(8)`): **Time `O(n)`** (với hệ số `8`), **Space `O(1)`** (mảng `8` phần tử).

**Nút thắt.** Làm đúng nhưng cồng kềnh. Ta đã thấy mô hình lặp lại **từng lớp `8` chữ**, có thể tính trực tiếp chi phí của chữ thứ `i` mà không cần mô phỏng.

## 6. Ý tưởng thuật toán / bước đột phá

Khi gán chữ lần lượt cho phím ít chữ nhất, ta đang **lấp đầy từng lớp 8 vị trí**: 8 chữ đầu vào vị trí `1` của 8 phím, 8 chữ tiếp theo vào vị trí `2`, ... Vậy chữ thứ `i` (đếm từ `0`) luôn rơi vào vị trí **`i // 8 + 1`**, tức tốn đúng `i // 8 + 1` lần bấm:

> **đáp án = Σ (i // 8 + 1) với `i = 0, 1, ..., n - 1`**

Không cần mô phỏng phím nào, và vì các chữ cái đôi một khác nhau nên **chỉ cần `n = len(word)`**. Ta có thể cộng theo từng chữ, theo từng lớp 8 chữ, hoặc dùng **công thức đóng** `4 · full(full + 1) + rem · (full + 1)`.

## 7. Các cách tiếp theo

### Cách 2 — Cộng theo từng lớp (tối đa 8 chữ mỗi lớp)

**Giải thích.** `level = 1`. Còn `n` chữ chưa tính: lấy `take = min(8, n)` chữ vào lớp hiện tại, mỗi chữ tốn `level`, cộng `take * level`, trừ `n -= take`, rồi `level += 1`. Lặp đến khi hết chữ. Cách này **hiện rõ** cấu trúc "lớp 1, lớp 2, ...".

In [10]:
def minimum_pushes_levels(word: str) -> int:
    n = len(word)
    total = 0
    level = 1                                          # chi phí của lớp hiện tại
    while n > 0:
        take = min(8, n)                               # một lớp chứa tối đa 8 chữ (8 phím)
        total += take * level
        n -= take
        level += 1
    return total


for word, expected in examples:
    print(repr(word), "->", minimum_pushes_levels(word), "(kỳ vọng", expected, ")")

# Quan sát từng lớp với n = 20
n, level = 20, 1
while n > 0:
    take = min(8, n)
    print(f"lớp {level}: {take} chữ x {level} = {take * level}")
    n -= take
    level += 1

'abcde' -> 5 (kỳ vọng 5 )
'xycdefghij' -> 12 (kỳ vọng 12 )
'a' -> 1 (kỳ vọng 1 )
'abcdefgh' -> 8 (kỳ vọng 8 )
'abcdefghi' -> 10 (kỳ vọng 10 )
lớp 1: 8 chữ x 1 = 8
lớp 2: 8 chữ x 2 = 16
lớp 3: 4 chữ x 3 = 12


**Độ phức tạp Cách 2:** số lớp là `ceil(n / 8) <= 4` với `n <= 26`: **Time `O(n / 8)`**, **Space `O(1)`**.

### Cách 3 — Công thức đóng với `divmod`

**Giải thích.** `full, rem = divmod(n, 8)`. Các lớp đầy đủ cho `8 · (1 + 2 + ... + full) = 4 · full · (full + 1)`. Các chữ dư `rem` đều nằm ở lớp `full + 1` nên tốn `rem · (full + 1)`. Đáp án `4 · full · (full + 1) + rem · (full + 1)`, tính trong `O(1)`.

In [11]:
def minimum_pushes_formula(word: str) -> int:
    full, rem = divmod(len(word), 8)                   # full lớp đầy đủ, rem chữ dư
    return 4 * full * (full + 1) + rem * (full + 1)


for word, expected in examples:
    print(repr(word), "->", minimum_pushes_formula(word), "(kỳ vọng", expected, ")")

# Kiểm tra công thức với n = 1..26 so với tổng trực tiếp
for n in range(1, 27):
    w = "a" * n                                        # chỉ cần độ dài; công thức dùng len
    assert minimum_pushes_formula(w) == sum(i // 8 + 1 for i in range(n))
print("công thức khớp tổng trực tiếp với n = 1..26")

'abcde' -> 5 (kỳ vọng 5 )
'xycdefghij' -> 12 (kỳ vọng 12 )
'a' -> 1 (kỳ vọng 1 )
'abcdefgh' -> 8 (kỳ vọng 8 )
'abcdefghi' -> 10 (kỳ vọng 10 )
công thức khớp tổng trực tiếp với n = 1..26


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Một biểu thức `sum(i // 8 + 1 ...)`

**Giải thích.** Cách ngắn nhất, thể hiện trực tiếp công thức mục 6: `sum(i // 8 + 1 for i in range(len(word)))`.

In [12]:
def minimum_pushes_sum(word: str) -> int:
    return sum(i // 8 + 1 for i in range(len(word)))


for word, expected in examples:
    print(repr(word), "->", minimum_pushes_sum(word), "(kỳ vọng", expected, ")")

'abcde' -> 5 (kỳ vọng 5 )
'xycdefghij' -> 12 (kỳ vọng 12 )
'a' -> 1 (kỳ vọng 1 )
'abcdefgh' -> 8 (kỳ vọng 8 )
'abcdefghi' -> 10 (kỳ vọng 10 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (generator, không tạo danh sách).

In [13]:
# Code nộp LeetCode cho Cách 4 (ngắn, đúng công thức)
class Solution:
    def minimumPushes(self, word: str) -> int:
        return sum(i // 8 + 1 for i in range(len(word)))

## 8. TRACE từng bước

Ta theo dõi (a) Cách 4 với `n = 20` chữ (từng chữ tốn bao nhiêu) và (b) Cách 2 (theo lớp).

In [14]:
def trace_sum(word):
    n = len(word)
    total = 0
    print(f"word độ dài {n}")
    for i in range(n):
        cost = i // 8 + 1
        total += cost
        print(f"chữ thứ i={i:>2}: i // 8 + 1 = {i} // 8 + 1 = {cost} | tổng = {total}")
    print("=> kết quả:", total)
    return total


assert trace_sum("abcdefghijklmnopqrst") == 8 * 1 + 8 * 2 + 4 * 3 == 36

word độ dài 20
chữ thứ i= 0: i // 8 + 1 = 0 // 8 + 1 = 1 | tổng = 1
chữ thứ i= 1: i // 8 + 1 = 1 // 8 + 1 = 1 | tổng = 2
chữ thứ i= 2: i // 8 + 1 = 2 // 8 + 1 = 1 | tổng = 3
chữ thứ i= 3: i // 8 + 1 = 3 // 8 + 1 = 1 | tổng = 4
chữ thứ i= 4: i // 8 + 1 = 4 // 8 + 1 = 1 | tổng = 5
chữ thứ i= 5: i // 8 + 1 = 5 // 8 + 1 = 1 | tổng = 6
chữ thứ i= 6: i // 8 + 1 = 6 // 8 + 1 = 1 | tổng = 7
chữ thứ i= 7: i // 8 + 1 = 7 // 8 + 1 = 1 | tổng = 8
chữ thứ i= 8: i // 8 + 1 = 8 // 8 + 1 = 2 | tổng = 10
chữ thứ i= 9: i // 8 + 1 = 9 // 8 + 1 = 2 | tổng = 12
chữ thứ i=10: i // 8 + 1 = 10 // 8 + 1 = 2 | tổng = 14
chữ thứ i=11: i // 8 + 1 = 11 // 8 + 1 = 2 | tổng = 16
chữ thứ i=12: i // 8 + 1 = 12 // 8 + 1 = 2 | tổng = 18
chữ thứ i=13: i // 8 + 1 = 13 // 8 + 1 = 2 | tổng = 20
chữ thứ i=14: i // 8 + 1 = 14 // 8 + 1 = 2 | tổng = 22
chữ thứ i=15: i // 8 + 1 = 15 // 8 + 1 = 2 | tổng = 24
chữ thứ i=16: i // 8 + 1 = 16 // 8 + 1 = 3 | tổng = 27
chữ thứ i=17: i // 8 + 1 = 17 // 8 + 1 = 3 | tổng = 30
chữ thứ i=18:

In [15]:
def trace_levels(word):
    n, total, level = len(word), 0, 1
    print(f"n = {n}")
    while n > 0:
        take = min(8, n)
        total += take * level
        print(f"lớp {level}: lấy {take} chữ, mỗi chữ tốn {level} -> +{take * level} | tổng = {total}, còn {n - take} chữ")
        n -= take
        level += 1
    return total


assert trace_levels("abcdefghijklmnopqrstuvwxyz") == 8 + 16 + 24 + 2 * 4 == 56

n = 26
lớp 1: lấy 8 chữ, mỗi chữ tốn 1 -> +8 | tổng = 8, còn 18 chữ
lớp 2: lấy 8 chữ, mỗi chữ tốn 2 -> +16 | tổng = 24, còn 10 chữ
lớp 3: lấy 8 chữ, mỗi chữ tốn 3 -> +24 | tổng = 48, còn 2 chữ
lớp 4: lấy 2 chữ, mỗi chữ tốn 4 -> +8 | tổng = 56, còn 0 chữ


Bảng tóm tắt (`n = 26`):

| lớp (chi phí mỗi chữ) | số chữ | chi phí |
|---|---|---|
| 1 | 8 | 8 |
| 2 | 8 | 16 |
| 3 | 8 | 24 |
| 4 | 2 | 8 |
| **tổng** | 26 | **56** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (một biểu thức `sum`). Đúng class `Solution`, đúng method `minimumPushes(self, word: str) -> int`.

In [16]:
class Solution:
    def minimumPushes(self, word: str) -> int:
        # chữ thứ i (đếm từ 0) nằm ở lớp i // 8 của 8 phím, tốn i // 8 + 1 lần bấm
        return sum(i // 8 + 1 for i in range(len(word)))


sol = Solution()
for word, expected in examples:
    print(repr(word), "->", sol.minimumPushes(word))

'abcde' -> 5
'xycdefghij' -> 12
'a' -> 1
'abcdefgh' -> 8
'abcdefghi' -> 10


## 10. Vì sao thuật toán đúng

- **Mô hình chi phí.** Một phím chứa `m` chữ tốn `1 + 2 + ... + m = m(m + 1) / 2`. Gọi `m_1, ..., m_8` là số chữ trên `8` phím, `Σ m_k = n`. Cần cực tiểu `Σ m_k(m_k + 1) / 2`.
- **Hàm lồi.** `f(m) = m(m + 1) / 2` là hàm **lồi**: `f(m + 1) - f(m) = m + 1` tăng theo `m`. Với tổng `n` cố định, tổng `Σ f(m_k)` nhỏ nhất khi các `m_k` **chênh nhau nhiều nhất là `1`** (chia đều), vì chuyển một chữ từ phím đông sang phím vắng luôn không làm tăng tổng.
- **Chia đều = lấp đầy theo lớp.** Chia đều nghĩa là `n % 8` phím có `n // 8 + 1` chữ và các phím còn lại có `n // 8` chữ. Chi phí là `Σ (i // 8 + 1)` trên `i = 0..n - 1`.
- **Độc lập với chữ cái.** Vì các chữ cái đôi một khác nhau và mỗi chữ gõ **một lần**, chi phí gõ chữ ở vị trí `k` luôn `k`; đáp án chỉ phụ thuộc `n`.
- **Các cách tương đương.** Mô phỏng (Cách 1), theo lớp (Cách 2), công thức (Cách 3) và `sum` (Cách 4) cùng tính `Σ (i // 8 + 1)`; ta kiểm chứng với một quy hoạch động vét cạn ở phần test.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word)`.

- **Cách 1 (mô phỏng 8 phím):** Time `O(n)` (hệ số `8`), Space `O(1)`.
- **Cách 2 (theo lớp):** Time `O(n / 8)`, Space `O(1)`.
- **Cách 3 (công thức `divmod`):** Time `O(1)`, Space `O(1)`.
- **Cách 4 (`sum` generator):** Time `O(n)`, Space `O(1)`.
- Với `n <= 26` mọi cách chạy tức thì. Việc đáng nhớ là mô hình "lớp 8 chữ" và hàm lồi.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "simulate": minimum_pushes_simulate,
    "levels": minimum_pushes_levels,
    "formula": minimum_pushes_formula,
    "sum": minimum_pushes_sum,
    "submit": sol.minimumPushes,
}

for word, expected in examples:
    for key, fn in impls.items():
        got = fn(word)
        assert got == expected, (key, word, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
ALPHABET = "abcdefghijklmnopqrstuvwxyz"

def check_all(n, expected, label):
    word = ALPHABET[:n]                                  # n chữ cái đôi một khác nhau
    for key, fn in impls.items():
        got = fn(word)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all(1, 1, "một chữ cái")
check_all(2, 2, "hai chữ cái: hai phím khác nhau")
check_all(7, 7, "n = 7 < 8: mỗi chữ một phím")
check_all(8, 8, "n = 8: đúng lấp đầy lớp 1")
check_all(9, 10, "n = 9: chữ thứ 9 phải là chữ thứ hai trên một phím")
check_all(16, 24, "n = 16: hai lớp đầy đủ: 8 + 16")
check_all(17, 27, "n = 17: thêm một chữ ở lớp 3")
check_all(24, 48, "n = 24: ba lớp đầy đủ: 8 + 16 + 24")
check_all(25, 52, "n = 25: thêm một chữ ở lớp 4")
check_all(26, 56, "n = 26 (tối đa): 8 + 16 + 24 + 8")

PASS một chữ cái -> 1
PASS hai chữ cái: hai phím khác nhau -> 2
PASS n = 7 < 8: mỗi chữ một phím -> 7
PASS n = 8: đúng lấp đầy lớp 1 -> 8
PASS n = 9: chữ thứ 9 phải là chữ thứ hai trên một phím -> 10
PASS n = 16: hai lớp đầy đủ: 8 + 16 -> 24
PASS n = 17: thêm một chữ ở lớp 3 -> 27
PASS n = 24: ba lớp đầy đủ: 8 + 16 + 24 -> 48
PASS n = 25: thêm một chữ ở lớp 4 -> 52
PASS n = 26 (tối đa): 8 + 16 + 24 + 8 -> 56


### 12.2. Tính chất và phản ví dụ

In [19]:
# Chuẩn vàng: quy hoạch động vét cạn mọi cách chia n chữ cho 8 phím
# dp[k][m] = chi phí nhỏ nhất khi chia m chữ cho k phím; một phím chứa j chữ tốn j(j + 1)/2
def dp_min_cost(n, keys=8):
    INF = float("inf")
    dp = [[INF] * (n + 1) for _ in range(keys + 1)]
    dp[0][0] = 0
    for k in range(1, keys + 1):
        for m in range(n + 1):
            for j in range(m + 1):
                if dp[k - 1][m - j] + j * (j + 1) // 2 < dp[k][m]:
                    dp[k][m] = dp[k - 1][m - j] + j * (j + 1) // 2
    return dp[keys][n]

for n in range(1, 27):
    assert dp_min_cost(n) == sol.minimumPushes(ALPHABET[:n]), n
print("PASS quy hoạch động vét cạn (mọi cách chia chữ cho 8 phím) khớp đáp án với n = 1..26")

# Tính chất 1: đáp án chỉ phụ thuộc độ dài (thử với các chuỗi ngẫu nhiên có chữ cái khác nhau)
random.seed(11)
for _ in range(300):
    n = random.randint(1, 26)
    w = "".join(random.sample(ALPHABET, n))
    assert sol.minimumPushes(w) == sol.minimumPushes(ALPHABET[:n])
print("PASS đáp án chỉ phụ thuộc độ dài, không phụ thuộc chữ cái")

# Tính chất 2: đơn điệu tăng theo n, và hiệu liên tiếp là i // 8 + 1
vals = [sol.minimumPushes(ALPHABET[:n]) for n in range(1, 27)]
assert all(vals[i + 1] - vals[i] == (i + 1) // 8 + 1 for i in range(25))
assert all(vals[i] < vals[i + 1] for i in range(25))
print("PASS tăng nghiêm ngặt; chữ thứ n + 1 tốn n // 8 + 1")

# Tính chất 3: đáp án nằm giữa n (mọi chữ 1 lần) và n(n+1)/2 (dồn hết vào một phím)
for n in range(1, 27):
    v = sol.minimumPushes(ALPHABET[:n])
    assert n <= v <= n * (n + 1) // 2
print("PASS n <= đáp án <= n(n + 1) / 2")

# Phản ví dụ 1: dùng 10 phím (nhầm với bàn phím 0-9)
def sai_10_phim(word):
    return sum(i // 10 + 1 for i in range(len(word)))

assert sai_10_phim("xycdefghij") == 10 != 12
print("PASS phản ví dụ: 10 phím cho", sai_10_phim("xycdefghij"), "thay vì 12")

# Phản ví dụ 2: quên + 1
def sai_khong_cong_1(word):
    return sum(i // 8 for i in range(len(word)))

assert sai_khong_cong_1("abcde") == 0 != 5
print("PASS phản ví dụ: i // 8 cho", sai_khong_cong_1("abcde"), "thay vì 5")

PASS quy hoạch động vét cạn (mọi cách chia chữ cho 8 phím) khớp đáp án với n = 1..26
PASS đáp án chỉ phụ thuộc độ dài, không phụ thuộc chữ cái
PASS tăng nghiêm ngặt; chữ thứ n + 1 tốn n // 8 + 1
PASS n <= đáp án <= n(n + 1) / 2
PASS phản ví dụ: 10 phím cho 10 thay vì 12
PASS phản ví dụ: i // 8 cho 0 thay vì 5


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(3014)
DP_REF = {n: dp_min_cost(n) for n in range(1, 27)}        # tính quy hoạch động một lần cho mọi n
for trial in range(2000):
    n = random.randint(1, 26)
    word = "".join(random.sample(ALPHABET, n))
    ref = DP_REF[n]
    for key, fn in impls.items():
        assert fn(word) == ref, (key, word)
print("PASS 2000 bộ ngẫu nhiên - mọi cách khớp quy hoạch động vét cạn")

PASS 2000 bộ ngẫu nhiên - mọi cách khớp quy hoạch động vét cạn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng 10 phím (hoặc 9 phím) thay vì 8

In [21]:
def sai_so_phim(word):
    return sum(i // 10 + 1 for i in range(len(word)))     # tưởng có 10 phím (0-9)


print("sai  :", sai_so_phim("xycdefghij"), "| đúng:", Solution().minimumPushes("xycdefghij"))

sai  : 10 | đúng: 12


**Vì sao sai:** đề nói các phím được đánh số từ `2` đến `9`, tức **8 phím** (phím `1`, `*`, `0`, `#` không dùng để gán chữ). Dùng `10` thì chữ thứ `9` và `10` vẫn bị tính là vị trí `1` (sai), cho `10` thay vì `12`.

### Lỗi 2: Quên `+ 1` (chữ đầu tiên tốn 1 lần bấm)

In [22]:
def sai_thieu_1(word):
    return sum(i // 8 for i in range(len(word)))


print("sai  :", sai_thieu_1("abcde"), "| đúng:", Solution().minimumPushes("abcde"))

sai  : 0 | đúng: 5


**Vì sao sai:** chữ ở vị trí đầu tiên của một phím cần **`1`** lần bấm (không phải `0`). Khi đếm từ `0` bằng `i // 8` ta được "số lớp trước đó", còn chi phí là `i // 8 + 1`.

### Lỗi 3: Chỉ xử lý `n <= 8`

In [23]:
def sai_chi_8(word):
    return len(word)                                      # đúng khi n <= 8


print("sai  :", sai_chi_8("xycdefghij"), "| đúng:", Solution().minimumPushes("xycdefghij"))

sai  : 10 | đúng: 12


**Vì sao sai:** khi `n > 8`, `n - 8` chữ buộc phải làm chữ thứ hai, thứ ba, ... trên các phím nên tốn `2`, `3`, ... lần bấm. `return len(word)` chỉ đúng đến `n = 8`.

### Lỗi 4: Dùng `ceil` làm lệch một đơn vị

In [24]:
from math import ceil

def sai_ceil(word):
    return sum(ceil(i / 8) for i in range(len(word)))     # ceil(i / 8): i = 8 cho 1, nhưng cần 2


print("sai  :", sai_ceil("abcdefghi"), "| đúng:", Solution().minimumPushes("abcdefghi"))

sai  : 8 | đúng: 10


**Vì sao sai:** chữ thứ `i` (đếm từ `0`) tốn `floor(i / 8) + 1 = ceil((i + 1) / 8)`, **không** phải `ceil(i / 8)`. Với `i = 0` cho `0`, và với `i = 8` (chữ thứ `9`) cho `1` trong khi cần `2`. Quy tắc nhớ: **`i // 8 + 1`** khi đếm từ `0` (hoặc `ceil(i / 8)` khi đếm từ `1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "gán các phần tử vào các nhóm, chi phí tăng dần theo vị trí trong nhóm" → **chia đều** (hàm chi phí lồi) và công thức `i // số_nhóm + 1`.
- Công thức nhớ: **chữ thứ `i` (từ `0`) tốn `i // 8 + 1`**; tổng `sum(i // 8 + 1 for i in range(n))`.
- `divmod(n, 8)` cho `(số lớp đầy đủ, số chữ dư)`; tổng lớp đầy đủ `4 · full · (full + 1)`.
- Khi chữ cái **có thể lặp** (bản II): đếm tần suất, sắp **giảm dần**, rồi nhân tần suất `k`-lớn-nhất với chi phí `k // 8 + 1` (chữ hay gặp đặt vào vị trí rẻ).
- Đếm đúng số phím: phím `2` đến `9` là **8** phím, không phải 9 hay 10.
- Hàm chi phí lồi → "dàn đều" thường tối ưu; nhớ kiểm tra bằng quy hoạch động nhỏ.

## 15. Biến thể của bài

1. **Bản II** (chữ cái có thể lặp): sắp xếp theo tần suất giảm dần; chữ có tần suất cao đặt vào vị trí rẻ nhất.
2. Số phím **`K`** tổng quát thay vì `8`.
3. Mỗi chữ cái gõ **`c` lần** cho mọi chữ (chi phí nhân `c`).
4. Trả về **một cách gán cụ thể** (danh sách chữ cái trên từng phím).
5. Số phím tối thiểu cần dùng để đạt chi phí nhỏ nhất.

Thử biến thể 1, 2 và 4:

In [25]:
from collections import Counter

def minimum_pushes_ii(word):
    freq = sorted(Counter(word).values(), reverse=True)       # tần suất giảm dần
    return sum(f * (i // 8 + 1) for i, f in enumerate(freq))


def minimum_pushes_k_keys(word, K):
    return sum(i // K + 1 for i in range(len(word)))


def assign_keys(word):
    keys = [[] for _ in range(8)]
    for i, ch in enumerate(word):
        keys[i % 8].append(ch)                                 # chữ thứ i vào phím i % 8
    return keys


print("1. bản II với 'aabbbccd':", minimum_pushes_ii("aabbbccd"))        # 8 chữ khác 4 loại: freq [3,2,2,1] -> 8 phím, mỗi loại vị trí 1 -> 8
print("2. K = 4, n = 10:", minimum_pushes_k_keys("a" * 10, 4))             # 4*1 + 4*2 + 2*3 = 18
print("4.", assign_keys("abcdefghij"))
assert minimum_pushes_ii("aabbbccd") == 8
assert minimum_pushes_ii("a" * 5) == 5
assert minimum_pushes_k_keys("a" * 10, 4) == 18
assert minimum_pushes_k_keys("a" * 10, 8) == Solution().minimumPushes("abcdefghij")
assert assign_keys("abcdefghij") == [["a", "i"], ["b", "j"], ["c"], ["d"], ["e"], ["f"], ["g"], ["h"]]

1. bản II với 'aabbbccd': 8
2. K = 4, n = 10: 18
4. [['a', 'i'], ['b', 'j'], ['c'], ['d'], ['e'], ['f'], ['g'], ['h']]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng gán chữ vào phím ít chữ nhất | `size[k]`, `min`/`index` | `O(n)` | `O(1)` | hiểu bản chất "chia đều" |
| 2. Cộng theo từng lớp 8 chữ | `take = min(8, n)`, `level += 1` | `O(n/8)` | `O(1)` | thấy rõ cấu trúc lớp |
| 3. Công thức `divmod` | `4·full·(full+1) + rem·(full+1)` | `O(1)` | `O(1)` | cần `O(1)`, bản toán học |
| 4. `sum(i // 8 + 1 ...)` | một biểu thức | `O(n)` | `O(1)` | **cách nộp bài**, ngắn và rõ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `divmod(19, 8)`, `[i // 8 + 1 for i in range(10)]`, `Solution().minimumPushes("abcdefgh")`, `Solution().minimumPushes("abcdefghijklmnop")`, `Solution().minimumPushes("a")`.

**Bài 2 (tính tay).** Với `n = 21` chữ cái: chia thành các lớp (bao nhiêu chữ mỗi lớp, chi phí mỗi chữ) và tính tổng bằng cả hai cách: cộng theo lớp và công thức `4 · full · (full + 1) + rem · (full + 1)`.

**Bài 3 (điền TODO).** Hoàn thành `minimum_pushes_todo(word)` theo Cách 2 (theo lớp).

In [26]:
def minimum_pushes_todo(word):
    n = len(word)
    total = 0
    level = 1
    while n > 0:
        # TODO 1: take = min(8, n)  (số chữ trong lớp hiện tại)
        # TODO 2: total += take * level
        # TODO 3: n -= take ; level += 1
        break                                    # (xóa dòng này khi bạn điền xong)
    return total

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_pushes_formula2(word)` theo **công thức đóng** (Cách 3, `divmod`) mà không nhìn code mẫu. Kiểm tra với năm ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `minimum_pushes_ii2(word)` cho **bản II** (chữ cái có thể lặp, `1 <= len <= 10^5`): sắp xếp tần suất giảm dần rồi nhân với chi phí lớp. (b) Viết `minimum_pushes_keys(word, K)` với `K` phím. (c) Viết `largest_n_within_cost(limit)` trả về **số chữ lớn nhất** `n` (`<= 26`) mà `minimumPushes` của `n` chữ khác nhau **không vượt** `limit`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(2, 3)`, `[1, 1, 1, 1, 1, 1, 1, 1, 2, 2]`, `8`, `24` (`8 + 16`), `1`. Bài 3: ba TODO nằm trong comment; nhớ xóa dòng `break` khi đã điền xong.
- **Mức 2 (vừa):** Bài 2: `n = 21`: lớp 1: `8` chữ x `1` = `8`; lớp 2: `8` chữ x `2` = `16`; lớp 3: `5` chữ x `3` = `15`; tổng `39`. Công thức: `full = 2`, `rem = 5`: `4 · 2 · 3 + 5 · 3 = 24 + 15 = 39`. Bài 4: `full, rem = divmod(len(word), 8)`.
- **Mức 3 (sâu):** Bài 5(a): `freq = sorted(Counter(word).values(), reverse=True)`, rồi `sum(f * (i // 8 + 1) for i, f in enumerate(freq))`. 5(b): thay `8` bằng `K`. 5(c): thử `n` từ `26` giảm xuống `0`, lấy `n` lớn nhất có `sum(i // 8 + 1 for i in range(n)) <= limit` (hoặc dùng `accumulate`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert divmod(19, 8) == (2, 3)
assert [i // 8 + 1 for i in range(10)] == [1, 1, 1, 1, 1, 1, 1, 1, 2, 2]
assert Solution().minimumPushes("abcdefgh") == 8
assert Solution().minimumPushes("abcdefghijklmnop") == 24
assert Solution().minimumPushes("a") == 1

# Bài 2
assert sum(i // 8 + 1 for i in range(21)) == 39
assert 4 * 2 * 3 + 5 * 3 == 39
assert Solution().minimumPushes(ALPHABET[:21]) == 39

# Bài 3
def minimum_pushes_todo_ans(word):
    n = len(word)
    total = 0
    level = 1
    while n > 0:
        take = min(8, n)
        total += take * level
        n -= take
        level += 1
    return total

for word, expected in examples:
    assert minimum_pushes_todo_ans(word) == expected

# Bài 4
def minimum_pushes_formula2(word):
    full, rem = divmod(len(word), 8)
    return 4 * full * (full + 1) + rem * (full + 1)

for word, expected in examples:
    assert minimum_pushes_formula2(word) == expected

# Bài 5(a)
def minimum_pushes_ii2(word):
    freq = sorted(Counter(word).values(), reverse=True)
    return sum(f * (i // 8 + 1) for i, f in enumerate(freq))

assert minimum_pushes_ii2("abcde") == 5
assert minimum_pushes_ii2("xyzxyzxyzxyz") == 12
assert minimum_pushes_ii2("aabbbccd" * 3) == 24 * 1

# Bài 5(b)
def minimum_pushes_keys(word, K):
    return sum(i // K + 1 for i in range(len(word)))

assert minimum_pushes_keys("abcdefgh", 8) == 8
assert minimum_pushes_keys("abcdefgh", 2) == 2 * 1 + 2 * 2 + 2 * 3 + 2 * 4
assert minimum_pushes_keys("abcdefghijklmnopqrstuvwxyz", 8) == Solution().minimumPushes(ALPHABET)

# Bài 5(c)
def largest_n_within_cost(limit):
    best = 0
    for n in range(0, 27):
        if sum(i // 8 + 1 for i in range(n)) <= limit:
            best = n
    return best

assert largest_n_within_cost(0) == 0
assert largest_n_within_cost(8) == 8
assert largest_n_within_cost(9) == 8
assert largest_n_within_cost(10) == 9
assert largest_n_within_cost(1000) == 26
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chữ thứ `i` tốn `i // 8 + 1` lần bấm)

```python
class Solution:
    def minimumPushes(self, word: str) -> int:
        return sum(i // 8 + 1 for i in range(len(word)))
```

**Checklist**

- [ ] Có đúng **8** phím (`2` đến `9`)?
- [ ] Chi phí chữ thứ `i` (đếm từ `0`) là `i // 8 + 1` (có `+ 1`)?
- [ ] Hiểu vì sao chỉ cần `len(word)` (các chữ cái đôi một khác nhau, mỗi chữ gõ một lần)?
- [ ] Đã xử lý `n > 8` (chữ thứ hai, thứ ba trên một phím tốn `2`, `3` lần bấm)?
- [ ] Đã thử: `n = 1`, `n = 8`, `n = 9`, `n = 16`, `n = 26` (tối đa)?
- [ ] Skeleton đúng `class Solution` / `def minimumPushes(self, word: str) -> int` (không cần `from typing`)?